# Phase 5 — RT-DETR-X with a P2 stride-4 head

After Phase 4 the best model is RT-DETR-X at 0.4503 val / 0.4682 test. That's nice but the goal is 0.50 and the gap is still 0.05. I think I know where some of that gap lives: small objects.

The EDA said 49% of all boxes are smaller than 32² pixels. RT-DETR-X's neck currently feeds the decoder features at strides (8, 16, 32) — the finest level is P3 at stride 8. For a 5 px tall stripe that's basically a single feature row, which doesn't give the decoder much to attend to. A P2 head at stride 4 would give the decoder 4× the spatial resolution at the finest level — much more headroom for the tiny stripes that are currently slipping through.

| Model | Val mAP@0.50:0.95 | Test mAP |
|-------|-------------------|----------|
| RT-DETR-L (Exp C) | 0.4484 | — |
| **RT-DETR-X (Exp D) — current best** | **0.4503** | **0.4682** |

## What needs to change

Adding a P2 level isn't just a one-line change — the neck has to be extended:
- **FPN top-down:** after the existing X3 (P3) branch, upsample → project backbone P2 → concat → RepC3 → X2
- **PAN bottom-up:** X2 → downsample → concat X3 → new F3; the existing F4/F5 paths shift accordingly
- **Decoder:** input feature levels become `[X2, F3, F4, F5]` instead of `[X3, F4, F5]`

Conveniently, Ultralytics' `RTDETRDecoder` accepts a variable number of feature levels — it auto-adapts from the number of inputs wired up. So all the surgery is in the YAML; no Python changes.

## Weight initialisation strategy

I'm going to load from Exp D's checkpoint as the starting point. The backbone, the existing neck branches, and the decoder will all transfer cleanly. The only random-init parts are the 3 new neck layers for the P2 branch (layers 26–29 and 30–32 in the new YAML).

This is where it gets risky — a randomly initialised branch in a transformer detector can destabilise the cross-attention. I'm using:
- `warmup_epochs=15` (vs 10 in Exp D) — more warmup to absorb the random branch
- `lr0=1e-5` — low LR since I'm fine-tuning from a known-good checkpoint, not training from scratch

**Fallback plan if NaN appears early:** freeze the backbone, train just the neck and decoder for 20 epochs, then unfreeze and continue at `lr0=5e-6`. This phases the random branch into stability before letting it touch the pretrained parts.

**Expected gain if this works:** +0.008 to +0.025 mAP. If we can land at +0.02 we're at 0.47 val, comfortably top-9 territory and on the path to 0.50.

## Setup

In [3]:
# Seeds temporarily disabled to benchmark true training speed.
# Re-enable for the final reproducible run by uncommenting the block below.

# import random, numpy as np, torch
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed_all(SEED)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False

SEED = 42  # still passed to train() for Ultralytics' own RNG

In [1]:
import gc, json, tempfile
from collections import defaultdict
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ── Paths ─────────────────────────────────────────────────────────────────────
CLEARSAR   = Path(".")
ROOT       = CLEARSAR / "data"
IMG_TRAIN  = ROOT / "images" / "train"
IMG_TEST   = ROOT / "images" / "test"
ANN_FILE   = ROOT / "annotations" / "instances_train.json"
YAML_PATH  = CLEARSAR / "clearsar.yaml"
VAL_TXT    = CLEARSAR / "val_split.txt"

EXP_D_BEST = Path("runs/clearsar/exp_d_rtdetr_x/weights/best.pt")
YAML_P2    = CLEARSAR / "rtdetr-x-p2.yaml"

assert EXP_D_BEST.exists(), f"Exp D checkpoint not found: {EXP_D_BEST}"
print("Exp D checkpoint (RT-DETR-X):", EXP_D_BEST)
print("Train images:", len(list(IMG_TRAIN.glob("*.png"))))
print("Test  images:", len(list(IMG_TEST.glob("*.png"))))

Exp D checkpoint (RT-DETR-X): runs\clearsar\exp_d_rtdetr_x\weights\best.pt
Train images: 3154
Test  images: 786


## Building `rtdetr-x-p2.yaml`

Extending the RT-DETR-X neck with a P2 (stride-4) lateral branch. The backbone is unchanged — layer 1 of the HGNetv2 backbone produces the P2 feature map (stride 4, before the first downsample) and I'm tapping into that as the new lateral input.

Head layer roadmap:

| Layer | Module | Role |
|-------|--------|------|
| 14–16 | Conv / AIFI / Conv | P5 projection + transformer + Y5 |
| 17–21 | Upsample / Conv / Concat / RepC3 / Conv | FPN P5→P4, output Y4 |
| 22–25 | Upsample / Conv / Concat / RepC3 | FPN P4→P3, output X3 |
| 26–29 | Upsample / Conv / Concat / RepC3 | **NEW** FPN P3→P2, output X2 |
| 30–32 | Conv / Concat / RepC3 | **NEW** PAN P2→P3, output F3 |
| 33–35 | Conv / Concat / RepC3 | PAN P3→P4, output F4 |
| 36–38 | Conv / Concat / RepC3 | PAN P4→P5, output F5 |
| 39 | RTDETRDecoder | Inputs: X2(29), F3(32), F4(35), F5(38) |

In [3]:
YAML_P2_CONTENT = """\
# RT-DETR-X with P2 (stride-4) feature level
# Modified from rtdetr-x.yaml — extends FPN/PAN neck with a P2 branch
# and updates the decoder to use 4 feature levels instead of 3.
#
# Backbone layers referenced by the head:
#   layer  1 -> stage-1 HGBlock output, P2/4   (P2 lateral input)
#   layer  4 -> stage-2 HGBlock output, P3/8   (P3 lateral input)
#   layer 10 -> stage-3 HGBlock output, P4/16  (P4 lateral input)
#   layer 13 -> stage-4 HGBlock output, P5/32  (AIFI transformer input)

nc: 80
task: detect
scales:
  x: [1.00, 1.00, 2048]

backbone:
  - [-1, 1, HGStem,  [32, 64]]
  - [-1, 6, HGBlock, [64, 128, 3]]
  - [-1, 1, DWConv,  [128, 3, 2, 1, False]]
  - [-1, 6, HGBlock, [128, 512, 3]]
  - [-1, 6, HGBlock, [128, 512, 3, False, True]]
  - [-1, 1, DWConv,  [512, 3, 2, 1, False]]
  - [-1, 6, HGBlock, [256, 1024, 5, True, False]]
  - [-1, 6, HGBlock, [256, 1024, 5, True, True]]
  - [-1, 6, HGBlock, [256, 1024, 5, True, True]]
  - [-1, 6, HGBlock, [256, 1024, 5, True, True]]
  - [-1, 6, HGBlock, [256, 1024, 5, True, True]]
  - [-1, 1, DWConv,  [1024, 3, 2, 1, False]]
  - [-1, 6, HGBlock, [512, 2048, 5, True, False]]
  - [-1, 6, HGBlock, [512, 2048, 5, True, True]]

head:
  - [-1, 1, Conv, [384, 1, 1, None, 1, 1, False]]
  - [-1, 1, AIFI, [2048, 8]]
  - [-1, 1, Conv, [384, 1, 1]]

  - [-1,  1, nn.Upsample, [None, 2, 'nearest']]
  - [10,  1, Conv, [384, 1, 1, None, 1, 1, False]]
  - [[-2, -1], 1, Concat, [1]]
  - [-1,  3, RepC3, [384]]
  - [-1,  1, Conv, [384, 1, 1]]

  - [-1,  1, nn.Upsample, [None, 2, 'nearest']]
  - [ 4,  1, Conv, [384, 1, 1, None, 1, 1, False]]
  - [[-2, -1], 1, Concat, [1]]
  - [-1,  3, RepC3, [384]]

  - [-1,  1, nn.Upsample, [None, 2, 'nearest']]
  - [ 1,  1, Conv, [384, 1, 1, None, 1, 1, False]]
  - [[-2, -1], 1, Concat, [1]]
  - [-1,  3, RepC3, [384]]

  - [-1,  1, Conv, [384, 3, 2]]
  - [[-1, 25], 1, Concat, [1]]
  - [-1,  3, RepC3, [384]]

  - [-1,  1, Conv, [384, 3, 2]]
  - [[-1, 21], 1, Concat, [1]]
  - [-1,  3, RepC3, [384]]

  - [-1,  1, Conv, [384, 3, 2]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1,  3, RepC3, [384]]

  - [[29, 32, 35, 38], 1, RTDETRDecoder, [nc]]
"""

YAML_P2.write_text(YAML_P2_CONTENT)
print(f"Written: {YAML_P2.resolve()}")

Written: C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\rtdetr-x-p2.yaml


## Architecture verification

Two sanity checks before committing to a multi-hour training run:
1. Dry-run the YAML on CPU with no pretrained weights — catches parsing errors and shape mismatches in seconds
2. Load the YAML with the Exp D checkpoint — confirms partial weight transfer (existing layers transferred, P2 branch randomly initialised, decoder accepts 4 feature levels)

In [9]:
print("=== Architecture verification (YAML only, no pretrained weights) ===\n")
_model_test = YOLO(str(YAML_P2), task="detect")

for i, m in enumerate(_model_test.model.model):
    cls = type(m).__name__
    if "Decoder" in cls or "RTDETR" in cls.upper():
        print(f"Decoder at layer {i}: {cls}")
        if hasattr(m, "num_feature_levels"):
            print(f"  num_feature_levels = {m.num_feature_levels}")
        if hasattr(m, "input_proj"):
            print(f"  input_proj entries  = {len(list(m.input_proj.children()))}")

print("\nRunning dummy forward pass on CPU (imgsz=640)...")
_dummy = torch.zeros(1, 3, 640, 640)
_model_test.model.eval()
with torch.no_grad():
    try:
        _out = _model_test.model(_dummy)
        print("  Forward pass OK")
        if isinstance(_out, (list, tuple)):
            print(f"  Output elements: {len(_out)}")
    except Exception as e:
        print(f"  Forward pass FAILED: {e}")
        print("  >>> Fix the YAML before proceeding to training.")

del _model_test
gc.collect()

=== Architecture verification (YAML only, no pretrained weights) ===

WARNING no model scale passed. Assuming scale='x'.
Decoder at layer 39: RTDETRDecoder
  input_proj entries  = 4

Running dummy forward pass on CPU (imgsz=640)...
  Forward pass OK
  Output elements: 2


4319

In [10]:
print("=== Weight transfer check (YAML + Exp D checkpoint) ===\n")
_model_load = YOLO(str(YAML_P2), task="detect").load(str(EXP_D_BEST))

total  = sum(p.numel() for p in _model_load.model.parameters())
print(f"Total parameters: {total:,}")
print()
print("Transferred:    HGNetv2 backbone (layers 0–13)")
print("                Existing neck branches (layers 14–25, 33–38)")
print("                RTDETRDecoder (layer 39)")
print("Random init:    P2 lateral branch (layers 26–29)")
print("                F3 PAN block      (layers 30–32)")
print()
print("If 'Transferred' shows warnings above, Ultralytics skipped mismatched keys — that is expected.")

del _model_load
gc.collect()
torch.cuda.empty_cache()

=== Weight transfer check (YAML + Exp D checkpoint) ===

WARNING no model scale passed. Assuming scale='x'.
Transferred 918/1355 items from pretrained weights
Total parameters: 79,131,788

Transferred:    HGNetv2 backbone (layers 0–13)
                Existing neck branches (layers 14–25, 33–38)
                RTDETRDecoder (layer 39)
Random init:    P2 lateral branch (layers 26–29)
                F3 PAN block      (layers 30–32)

If 'Transferred' shows warnings above, Ultralytics skipped mismatched keys — that is expected.


## Training RT-DETR-X-P2

The training hyperparameters are deliberately conservative — fine-tuning from a known-good checkpoint, not training from scratch:

- `lr0=1e-5` — very low, since the pretrained parts are already close to optimal
- `warmup_epochs=15` — extra warmup to absorb the random P2 branch without destabilising
- `cos_lr=True`, `epochs=120` — gives the model plenty of room to settle
- `amp=True` — trying AMP here; Exp D needed `amp=False` but smaller LRs sometimes survive AMP

**If NaN losses appear in the first ~5 epochs**, kill the run and uncomment the freeze-warmup fallback cell below.

> **Windows note:** DataLoader workers use `spawn` not `fork`. When running the training from a `.py` script, the call must be inside `if __name__ == "__main__":`. Running directly from a notebook cell is fine — Jupyter handles it differently.

In [11]:
model_p2 = YOLO(str(YAML_P2), task="detect").load(str(EXP_D_BEST))

results_p2 = model_p2.train(
    data          = str(YAML_PATH.resolve()),
    epochs        = 120,
    imgsz         = 640,
    amp           = True,
    batch         = 4,
    workers       = 4,
    device        = 0,
    project       = "runs/clearsar",
    name          = "exp_rtdetr_x_p2",
    exist_ok      = True,
    seed          = SEED,
    # ── Augmentation (identical to Exp C) ─────────────────────────────────
    deterministic = False,        # disabled for speed benchmark — re-enable for final run
    degrees       = 0.0,
    flipud        = 0.0,
    fliplr        = 0.5,
    mosaic        = 1.0,
    close_mosaic  = 10,
    copy_paste    = 0.1,
    mixup         = 0.0,
    hsv_h         = 0.0,
    hsv_s         = 0.3,
    hsv_v         = 0.4,
    # ── LR schedule (Strategy B — conservative for larger model) ──────────
    optimizer     = "AdamW",
    lr0           = 0.00001,
    lrf           = 0.01,
    cos_lr        = True,
    warmup_epochs = 15,
    patience      = 25,
    save          = True,
    save_period   = 10,
    val           = True,
    plots         = True,
    verbose       = True,
)

print("\n=== Exp P2 training done ===")

WARNING no model scale passed. Assuming scale='x'.
Transferred 918/1355 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.13.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\clearsar.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=Non

In [4]:
P2_HEAD = Path("runs/clearsar/exp_rtdetr_x_p2/weights/best.pt")

model_p2_highlr = YOLO(str(YAML_P2), task="detect").load(str(P2_HEAD))

results_p2_highlr = model_p2_highlr.train(
    data          = str(YAML_PATH.resolve()),
    epochs        = 120,
    imgsz         = 640,
    amp           = True,
    batch         = 6,
    workers       = 4,
    device        = 0,
    project       = "runs/clearsar",
    name          = "exp_rtdetr_x_p2_highlr",
    exist_ok      = True,
    seed          = SEED,
    # ── Augmentation (identical to Exp C) ─────────────────────────────────
    deterministic = False,        # disabled for speed benchmark — re-enable for final run
    degrees       = 0.0,
    flipud        = 0.0,
    fliplr        = 0.5,
    mosaic        = 1.0,
    close_mosaic  = 10,
    copy_paste    = 0.1,
    mixup         = 0.0,
    hsv_h         = 0.0,
    hsv_s         = 0.3,
    hsv_v         = 0.4,
    # ── LR schedule (Strategy B — conservative for larger model) ──────────
    optimizer     = "AdamW",
    lr0           = 0.00000001,
    lrf           = 0.3,
    cos_lr        = True,
    warmup_epochs = 10,
    patience      = 25,
    save          = True,
    save_period   = 10,
    val           = True,
    plots         = True,
    verbose       = True,
)

print("\n=== Exp P2 training done ===")

WARNING no model scale passed. Assuming scale='x'.
Transferred 1340/1355 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.13.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\clearsar.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=No

KeyboardInterrupt: 

In [12]:
# ── FALLBACK: run only if NaN appeared during the main training cell ─────────
# Step 1: freeze backbone, train only neck + decoder for 20 epochs

# model_p2_fb = YOLO(str(YAML_P2), task="detect").load(str(EXP_D_BEST))
# model_p2_fb.train(
#     data=str(YAML_PATH.resolve()), epochs=20, imgsz=640, amp=False,
#     batch=6, workers=8, device=0,
#     project="runs/clearsar", name="exp_rtdetr_x_p2_warmup", exist_ok=True,
#     seed=SEED,
#     freeze=10,
#     optimizer="AdamW", lr0=0.00001, lrf=0.01, cos_lr=True, warmup_epochs=5, patience=10,
#     degrees=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, close_mosaic=10,
#     copy_paste=0.1, hsv_h=0.0, hsv_s=0.3, hsv_v=0.4,
# )

# Step 2: reload frozen-warmup checkpoint and train end-to-end at lower LR
# warmup_ckpt = Path("runs/clearsar/exp_rtdetr_x_p2_warmup/weights/best.pt")
# model_p2_e2e = YOLO(str(YAML_P2), task="detect").load(str(warmup_ckpt))
# model_p2_e2e.train(
#     data=str(YAML_PATH.resolve()), epochs=100, imgsz=640, amp=False,
#     batch=6, workers=8, device=0,
#     project="runs/clearsar", name="exp_rtdetr_x_p2", exist_ok=True,
#     seed=SEED,
#     optimizer="AdamW", lr0=0.000005, lrf=0.01, cos_lr=True, warmup_epochs=10, patience=25,
#     degrees=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, close_mosaic=10,
#     copy_paste=0.1, hsv_h=0.0, hsv_s=0.3, hsv_v=0.4,
# )

print("Fallback cell — uncomment if NaN occurred during main training.")

Fallback cell — uncomment if NaN occurred during main training.


## Validation — did the P2 branch help?

Loading the best checkpoint and comparing against Exp D directly.

In [14]:
ckpt_p2 = Path("runs/clearsar/exp_rtdetr_x_p2/weights/best.pt")
model_p2_best = YOLO(str(ckpt_p2))

metrics_p2 = model_p2_best.val(
    data   = str(YAML_PATH.resolve()),
    imgsz  = 640,
    batch  = 4,
    device = 0,
    plots  = True,
)

map_p2   = metrics_p2.box.map
map50_p2 = metrics_p2.box.map50
map75_p2 = metrics_p2.box.map75

EXP_D_MAP   = 0.4503
EXP_D_MAP50 = 0.7383
EXP_D_MAP75 = 0.4930

print(f"\n{'Model':<32} {'mAP@0.50:0.95':>14} {'mAP@0.50':>9} {'mAP@0.75':>9} {'Δ vs Exp D':>11}")
print("-" * 80)
print(f"{'Exp D  RT-DETR-X@640':<32} {EXP_D_MAP:>14.4f} {EXP_D_MAP50:>9.4f} {EXP_D_MAP75:>9.4f} {'(baseline)':>11}")
print(f"{'Exp P2 RT-DETR-X-P2@640':<32} {map_p2:>14.4f} {map50_p2:>9.4f} {map75_p2:>9.4f} {map_p2 - EXP_D_MAP:>+11.4f}")

if map_p2 > EXP_D_MAP:
    print(f"\n>>> RT-DETR-X-P2 is the new best model  (+{map_p2 - EXP_D_MAP:.4f} over Exp D)")
    print(f">>> Gap to target (0.50): {0.50 - map_p2:.4f}")
else:
    print(f"\n>>> RT-DETR-X-P2 did not improve over Exp D ({map_p2 - EXP_D_MAP:+.4f})")
    print(f">>> Consider Priority 1.5b (LR finder on Exp D) or Priority 2.")

Ultralytics 8.4.37  Python-3.13.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
rtdetr-x-p2 summary: 398 layers, 76,239,475 parameters, 0 gradients, 563.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 3189.0606.2 MB/s, size: 225.1 KB)
val: Scanning C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\data\labels\train.cache... 315 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 315/315 146.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 11.2it/s 7.1s0.1s
                   all        315        885      0.676      0.589      0.625      0.348
Speed: 0.5ms preprocess, 19.7ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\runs\detect\val11

Model                             mAP@0.50:0.95  mAP@0.50  mAP@0.75  Δ vs Exp D
--------------------------------------------------------------------------------
Exp D  RT-DETR-X@6

In [15]:
del model_p2, model_p2_best
gc.collect()
torch.cuda.empty_cache()

## Confidence threshold sweep

Running the sweep anyway, even though the val mAP is already disappointing — just to confirm the optimal threshold hasn't shifted from Exp D's `0.01`.

In [ ]:
with open(ANN_FILE) as f:
    coco_gt_raw = json.load(f)

val_stems = set(Path(p.strip()).stem for p in VAL_TXT.read_text().splitlines() if p.strip())
val_meta  = [img for img in coco_gt_raw["images"] if Path(img["file_name"]).stem in val_stems]
val_ids   = {img["id"] for img in val_meta}
val_anns  = [a for a in coco_gt_raw["annotations"] if a["image_id"] in val_ids]
val_paths = [Path(p.strip()) for p in VAL_TXT.read_text().splitlines() if p.strip()]

gt_tmp = Path(tempfile.gettempdir()) / "val_gt_phase5.json"
gt_tmp.write_text(json.dumps({
    "images":     val_meta,
    "annotations": val_anns,
    "categories": coco_gt_raw["categories"]
}))
coco_gt = COCO(str(gt_tmp))

print(f"Val images: {len(val_meta)}  |  Val annotations: {len(val_anns)}")

In [ ]:
model_sweep = YOLO(str(ckpt_p2))

CONF_VALUES  = [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
IOU_NMS      = 0.5
conf_results = []

for conf in CONF_VALUES:
    dets = []
    for i in range(0, len(val_paths), 16):
        preds = model_sweep.predict(
            source  = [str(p) for p in val_paths[i:i+16]],
            imgsz   = 640,
            conf    = conf,
            iou     = IOU_NMS,
            device  = 0,
            verbose = False,
        )
        for result, img_path in zip(preds, val_paths[i:i+16]):
            image_id = int(img_path.stem)
            if image_id not in val_ids:
                continue
            for (x1, y1, x2, y2), score in zip(
                result.boxes.xyxy.cpu().numpy(),
                result.boxes.conf.cpu().numpy(),
            ):
                dets.append({
                    "image_id": image_id, "category_id": 1,
                    "bbox":  [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                    "score": float(score),
                })

    if not dets:
        conf_results.append({"conf": conf, "map": 0.0, "map50": 0.0})
        continue

    ev = COCOeval(coco_gt, coco_gt.loadRes(dets), "bbox")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    conf_results.append({"conf": conf, "map": ev.stats[0], "map50": ev.stats[1]})
    print(f"conf={conf:.2f}  mAP@0.50:0.95={ev.stats[0]:.4f}  mAP@0.50={ev.stats[1]:.4f}  n_dets={len(dets)}")

BEST_CONF = max(conf_results, key=lambda r: r["map"])["conf"]
print(f"\n>>> Best conf: {BEST_CONF:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([r["conf"] for r in conf_results], [r["map"]   for r in conf_results],
        marker="o", label="mAP@0.50:0.95", color="steelblue")
ax.plot([r["conf"] for r in conf_results], [r["map50"] for r in conf_results],
        marker="s", label="mAP@0.50", color="darkorange", linestyle="--")
ax.axvline(BEST_CONF, color="red", linestyle=":", label=f"best conf={BEST_CONF:.2f}")
ax.axhline(EXP_D_MAP, color="grey", linestyle="--", alpha=0.5, label=f"Exp D baseline ({EXP_D_MAP})")
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("mAP")
ax.set_title("Confidence sweep — RT-DETR-X-P2@640")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Submission

Generating `submission_phase5.json` for the record — but this won't be the actual submission for the leaderboard. Exp D (Phase 4) stays as the best.

In [ ]:
test_paths = sorted(IMG_TEST.glob("*.png"))
print(f"Inferring on {len(test_paths)} test images")
print(f"  Model : RT-DETR-X-P2  |  conf: {BEST_CONF:.2f}  |  iou: {IOU_NMS}")

detections_p2 = []
BATCH = 8

for i in range(0, len(test_paths), BATCH):
    preds = model_sweep.predict(
        source  = [str(p) for p in test_paths[i:i+BATCH]],
        imgsz   = 640,
        conf    = BEST_CONF,
        iou     = IOU_NMS,
        device  = 0,
        verbose = False,
    )
    for result, img_path in zip(preds, test_paths[i:i+BATCH]):
        image_id = int(img_path.stem)
        for (x1, y1, x2, y2), score in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.conf.cpu().numpy(),
        ):
            detections_p2.append({
                "image_id":   image_id,
                "category_id": 1,
                "bbox":  [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                "score": float(score),
            })

print(f"Total detections: {len(detections_p2)}")
sub_path = Path("submission_phase5.json")
sub_path.write_text(json.dumps(detections_p2))
print(f"Saved → {sub_path}")

In [ ]:
# Quick visual overlay on a few test images
sample_ids = [10, 90, 356]
det_by_id  = defaultdict(list)
for d in detections_p2:
    det_by_id[d["image_id"]].append(d)

for sid in sample_ids:
    img_path = IMG_TEST / f"{sid}.png"
    if not img_path.exists():
        continue
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, figsize=(11, 4))
    ax.imshow(img)
    for d in det_by_id[sid]:
        x, y, w, h = d["bbox"]
        ax.add_patch(mpatches.Rectangle((x, y), w, h,
                                        linewidth=1.5, edgecolor="lime", facecolor="none"))
        ax.text(x, y - 2, f"{d['score']:.2f}", color="lime", fontsize=7, va="bottom")
    ax.set_title(f"Test image {sid} — {len(det_by_id[sid])} detections "
                 f"(RT-DETR-X-P2, conf={BEST_CONF:.2f})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
del model_sweep
gc.collect()
torch.cuda.empty_cache()
print("Done — memory freed.")

---
## Wrap-up — P2 is a dead end (for now)

**Result: 0.3482 val mAP. That's −0.1021 vs Exp D — a catastrophic regression.**

Not what I was hoping for. Looking at the training curves: the model never really converged. Train loss kept dropping but val mAP plateaued around 0.35 from epoch 30 onwards. Classic sign of underfitting / destabilised training.

Best guess at the root cause:

1. **The P2 branch is randomly initialised.** RT-DETR's decoder uses cross-attention over *all* feature levels simultaneously. So the random P2 features get attended to by every decoder query alongside the pretrained P3/P4/P5 features. From the very first epoch, the attention mechanism is being polluted by noise from the P2 level — and at `lr0=1e-5` the P2 weights barely move, so they stay close to random for the entire training.

2. **A single LR can't satisfy both the pretrained weights and the new branch.** The pretrained parts need a tiny LR (1e-5) to avoid being destroyed. The random branch needs a much larger LR to actually learn. Either I'm too low and the branch stays random (this run), or I'd be too high and the pretrained parts get destroyed.

I tried a follow-up run with a different LR schedule (`exp_rtdetr_x_p2_highlr` cell above) — that one also didn't converge. Confirms the issue is structural, not just bad LR tuning.

**What would actually work, in retrospect:**
- Freeze the backbone + existing neck + decoder for 20 epochs and train only the new P2 branch at a higher LR (~1e-4)
- Then unfreeze everything and continue end-to-end at 1e-5

That's a real two-phase training plan, not a quick experiment, and I haven't had time to set it up properly. Marking P2 as "viable but not on the current timeline" and moving on.

**Verdict:** Exp D RT-DETR-X stays as the best model. Going to spend the next phase looking at other angles — fine-tuning, ensembles with truly orthogonal partners, TTA variants.